In [1]:
import jax.numpy as jnp
from jax import grad, jit, vmap
from jax import random
from jax import lax
from optimization import optimize_fire2
from potentials import effective_potential,effective_potential_all
import numpy as onp
from matplotlib import pyplot as plt

In [2]:
def forward3(w):
    def body(carry, w):
        conv = jnp.convolve(carry * w, kernel, mode='valid')
        out = jnp.zeros_like(w).at[1:-1].set(conv)
        return out, out
    
    init = jnp.ones(w.shape[0])
    kernel = jnp.ones(3)
    return jnp.vstack([init, lax.scan(body, jnp.ones(w.shape[0]), w.T)[1]]).T

In [11]:
width = 32
height = 64
key = random.key(0)
w = random.uniform(key, (height, width))

# p1s = random.uniform(key, (num_rods,3))



In [12]:
%time forward3(w).block_until_ready()
# CPU times: user 93.2 ms, sys: 2.96 ms, total: 96.1 ms
# Wall time: 94 ms

CPU times: user 99.1 ms, sys: 4.38 ms, total: 103 ms
Wall time: 170 ms


Array([[1.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 0.0000000e+00,
        0.0000000e+00, 0.0000000e+00],
       [1.0000000e+00, 5.3757513e-01, 3.6503273e-01, ..., 2.6597607e+04,
        4.0904156e+04, 3.5301363e+04],
       [1.0000000e+00, 1.0154681e+00, 3.9626214e-01, ..., 7.6378648e+04,
        4.9757328e+04, 1.2892481e+05],
       ...,
       [1.0000000e+00, 1.3469143e+00, 1.6578368e+00, ..., 2.4548130e+03,
        7.6722349e+03, 8.8550264e+03],
       [1.0000000e+00, 1.4892446e+00, 1.4656001e+00, ..., 7.2333270e+02,
        1.8449697e+03, 5.1617441e+03],
       [1.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 0.0000000e+00,
        0.0000000e+00, 0.0000000e+00]], dtype=float32)

In [14]:
rod_length = 1.0
dist = 0.5

num_rods = 10
# create jnp random array
key = random.key(0)
p1s = random.uniform(key, (num_rods,3))
key = random.key(1)
phi1 = random.uniform(key, (num_rods,1), minval=0., maxval=jnp.pi)
key = random.key(2)
theta1 = random.uniform(key, (num_rods,1), minval=0., maxval=2*jnp.pi)

# key = random.key(3)
# p2s = random.normal(key, (num_rods,3))
# key = random.key(4)
# phi2 = random.uniform(key, (num_rods,1), minval=0., maxval=jnp.pi)
# key = random.key(5)
# theta2 = random.uniform(key, (num_rods,1), minval=0., maxval=2*jnp.pi)

q0 = jnp.concatenate([p1s, phi1, theta1], axis=1)
# q0 = q0.flatten()
print(q0.shape)

from potentials import total_effective_potential


%time total_effective_potential(q0).block_until_ready()

(10, 5)
CPU times: user 159 ms, sys: 4 ms, total: 163 ms
Wall time: 156 ms


Array(13.995945, dtype=float32)

In [15]:
fval = total_effective_potential(q0)
print(f"f_val: {fval:.2f}")

f_val: 14.00


In [ ]:

total1 = 0
k = 0
for i in range(N):
    q_i = q0[i]
    for j in range(i+1, N):
        q_j = q0[j]
        # q_pairs[k,:] = jnp.concatenate([q_i, q_j])
        q_pairs = q_pairs.at[k].set(jnp.concatenate([q_i, q_j]))
        total1 += effective_potential(jnp.concatenate([q_i, q_j]))
        k += 1                        
    
end_time = time.time()
execution_time = end_time - start_time
print(f"Execution time (1): {execution_time:.2f} seconds")


start_time = time.time()
total2 = total_effective_potential(q_pairs)
end_time = time.time()
execution_time = end_time - start_time
print(f"Execution time (2): {execution_time:.2f} seconds")


print(total1)
print(total2)